# 10. Experimento 2: Robustez y Evaluación en Arranque en Frío (Cold-Start)

Este cuaderno implementa el segundo experimento adicional propuesto para el análisis de resultados del TFM.

El objetivo es evaluar cuantitativamente cómo responde el modelo **Two Towers** frente al problema del arranque en frío (Cold-Start) de clientes y productos, en comparación con el baseline de **Factorización de Matrices (SVD)** y un análisis de **Ablación de Atributos** (Feature Masking) en la Query Tower.

### Objetivos:
1. Cargar el catálogo, sets de entrenamiento (2024-2025) y validación (2026 Q1).
2. Identificar y segmentar los subconjuntos de arranque en frío en validación:
   - **Clientes Fríos (Cold Users)**: Clientes en validación que no registran compras en entrenamiento.
   - **Productos Fríos (Cold Items)**: Productos complementarios objetivos en validación que no se registran en entrenamiento.
3. Cargar el modelo base Two Towers entrenado y el baseline SVD.
4. Evaluar y comparar la recuperación (`Recall@K` y `MRR`) de ambos modelos sobre los segmentos fríos.
5. Realizar un estudio de **Ablación por Enmascaramiento** en inferencia (Zero-Shot Feature Masking) para medir la importancia de las variables de geolocalización (`LATITUD/LONGITUD`) y el ID del cliente (`RUC`).


In [1]:
import polars as pl
import numpy as np
import tensorflow as tf
import tensorflow_recommenders as tfrs
from surprise import Dataset, Reader, SVD
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Cambiar al directorio raiz si se ejecuta desde experimentos
if os.getcwd().endswith('experimentos'):
    os.chdir('..')
import time

# Configuración de reproducibilidad
np.random.seed(42)
tf.random.set_seed(42)
sns.set_theme(style="whitegrid")

print("TensorFlow version:", tf.__version__)
print("Polars version:", pl.__version__)


TensorFlow version: 2.21.0
Polars version: 1.41.2


## 1. Carga y Preparación de Datos


In [2]:
train_path = "data_processed/retrieval_train.parquet"
val_path = "data_processed/retrieval_val.parquet"
prd_path = "data_processed/products_catalog.parquet"

train_df = pl.read_parquet(train_path)
val_df = pl.read_parquet(val_path)
products_df = pl.read_parquet(prd_path).unique(subset=["id_producto"])

products_df = products_df.with_columns([
    pl.col("marca").fill_null("SIN_MARCA"),
    pl.col("familia1").fill_null("SIN_CATEGORIA"),
    pl.col("familia2").fill_null("SIN_SUBCATEGORIA"),
    pl.col("precio").fill_null(0.0),
    pl.col("peso_unitario").fill_null(0.0)
])

cand_cols_rename = {c: f"cand_{c}" if c != "id_producto" else c for c in products_df.columns}
products_renamed = products_df.rename(cand_cols_rename)

val_joined = val_df.join(
    products_renamed,
    left_on="COD_PROD_2",
    right_on="id_producto",
    how="left"
).rename({
    "cand_marca": "marca_2",
    "cand_familia1": "familia1_2",
    "cand_familia2": "familia2_2",
    "cand_precio": "precio_2",
    "cand_peso_unitario": "peso_unitario_2"
}).with_columns([
    pl.col("marca_2").fill_null("SIN_MARCA"),
    pl.col("familia1_2").fill_null("SIN_CATEGORIA"),
    pl.col("familia2_2").fill_null("SIN_SUBCATEGORIA"),
    pl.col("precio_2").fill_null(0.0),
    pl.col("peso_unitario_2").fill_null(0.0)
])


## 2. Segmentación de Clientes y Productos Fríos (Cold-Start)
Definimos e identificamos las entidades de validación que no se encuentran en los datos de entrenamiento.


In [3]:
# Clientes únicos en entrenamiento y validación
train_rucs = set(train_df["RUC"].unique().to_list())
val_rucs = set(val_joined["RUC"].unique().to_list())
cold_rucs = val_rucs - train_rucs

# Productos únicos en entrenamiento y validación
train_prods = set(train_df["COD_PROD_2"].unique().to_list())
val_prods = set(val_joined["COD_PROD_2"].unique().to_list())
cold_prods = val_prods - train_prods

print(f"Clientes Únicos Entrenamiento: {len(train_rucs):,}")
print(f"Clientes Únicos Validación: {len(val_rucs):,}")
print(f"Clientes Fríos (Cold Users): {len(cold_rucs):,} ({len(cold_rucs)/len(val_rucs)*100:.2f}% de val)")

print(f"\nProductos Únicos Entrenamiento: {len(train_prods):,}")
print(f"Productos Únicos Validación: {len(val_prods):,}")
print(f"Productos Fríos (Cold Items): {len(cold_prods):,} ({len(cold_prods)/len(val_prods)*100:.2f}% de val)")

# Filtrar sets de validación fríos
val_cold_users = val_joined.filter(pl.col("RUC").is_in(list(cold_rucs)))
val_cold_items = val_joined.filter(pl.col("COD_PROD_2").is_in(list(cold_prods)))

print(f"\nConsultas con Clientes Fríos: {val_cold_users.height:,}")
print(f"Consultas con Productos Fríos: {val_cold_items.height:,}")


Clientes Únicos Entrenamiento: 12,863
Clientes Únicos Validación: 8,245
Clientes Fríos (Cold Users): 776 (9.41% de val)

Productos Únicos Entrenamiento: 10,935
Productos Únicos Validación: 6,578
Productos Fríos (Cold Items): 483 (7.34% de val)



Consultas con Clientes Fríos: 78,086
Consultas con Productos Fríos: 82,384


## 3. Cargar el Modelo de Factorización de Matrices (Surprise SVD)


In [4]:
# Entrenar SVD sobre una muestra del 10% de entrenamiento (ratings implícitos por conteo)
train_subset_svd = train_df.sample(fraction=0.10, seed=42)
train_counts = train_subset_svd.group_by(["RUC", "COD_PROD_2"]).agg(
    pl.len().alias("rating")
).to_pandas()

reader = Reader(rating_scale=(1, train_counts["rating"].max()))
data = Dataset.load_from_df(train_counts[["RUC", "COD_PROD_2", "rating"]], reader)
trainset = data.build_full_trainset()

svd = SVD(n_factors=50, n_epochs=10, random_state=42)
t0 = time.time()
svd.fit(trainset)
print(f"SVD entrenado en {time.time() - t0:.2f} segundos.")


SVD entrenado en 13.03 segundos.


## 4. Cargar el Recomendador Two Towers Base (LogQ)


In [5]:
vocab_ruc = sorted(train_df["RUC"].unique().to_list())
vocab_ciudad = sorted(train_df["CIUDAD"].unique().to_list())
vocab_ruta = sorted(train_df["RUTA"].unique().to_list())
vocab_products = sorted(products_df["id_producto"].unique().to_list())
vocab_marca = sorted(products_df["marca"].unique().to_list())
vocab_familia1 = sorted(products_df["familia1"].unique().to_list())
vocab_familia2 = sorted(products_df["familia2"].unique().to_list())

class QueryTower(tf.keras.Model):
    def __init__(self, vocab_ruc, vocab_ciudad, vocab_ruta, vocab_products, embedding_dim=128, dropout_rate=0.2):
        super().__init__()
        self.ruc_lookup = tf.keras.layers.StringLookup(vocabulary=vocab_ruc, mask_token=None)
        self.ruc_embedding = tf.keras.layers.Embedding(len(vocab_ruc) + 1, 64, name="ruc_emb")
        
        self.ciudad_lookup = tf.keras.layers.StringLookup(vocabulary=vocab_ciudad, mask_token=None)
        self.ciudad_embedding = tf.keras.layers.Embedding(len(vocab_ciudad) + 1, 16, name="ciudad_emb")
        
        self.ruta_lookup = tf.keras.layers.StringLookup(vocabulary=vocab_ruta, mask_token=None)
        self.ruta_embedding = tf.keras.layers.Embedding(len(vocab_ruta) + 1, 32, name="ruta_emb")
        
        self.product_lookup = tf.keras.layers.StringLookup(vocabulary=vocab_products, mask_token=None)
        self.product_embedding = tf.keras.layers.Embedding(len(vocab_products) + 1, 64, name="product_emb")
        
        self.geo_normalization = tf.keras.layers.Normalization(axis=-1)
        
        self.mlp = tf.keras.Sequential([
            tf.keras.layers.Dense(256, activation="relu"),
            tf.keras.layers.Dropout(dropout_rate),
            tf.keras.layers.Dense(128, activation="relu"),
            tf.keras.layers.Dropout(dropout_rate),
            tf.keras.layers.Dense(embedding_dim, name="query_projection")
        ])
        
    def call(self, inputs):
        ruc_emb = self.ruc_embedding(self.ruc_lookup(inputs["RUC"]))
        ciudad_emb = self.ciudad_embedding(self.ciudad_lookup(inputs["CIUDAD"]))
        ruta_emb = self.ruta_embedding(self.ruta_lookup(inputs["RUTA"]))
        product_emb = self.product_embedding(self.product_lookup(inputs["COD_PROD"]))
        
        lat = tf.expand_dims(inputs["LATITUD"], axis=-1)
        lon = tf.expand_dims(inputs["LONGITUD"], axis=-1)
        geo_features = tf.concat([lat, lon], axis=-1)
        geo_norm = self.geo_normalization(geo_features)
        
        concatenated = tf.concat([ruc_emb, ciudad_emb, ruta_emb, product_emb, geo_norm], axis=-1)
        return tf.math.l2_normalize(self.mlp(concatenated), axis=-1)

class CandidateTower(tf.keras.Model):
    def __init__(self, vocab_products, vocab_marca, vocab_familia1, vocab_familia2, embedding_dim=128, dropout_rate=0.2):
        super().__init__()
        self.product_lookup = tf.keras.layers.StringLookup(vocabulary=vocab_products, mask_token=None)
        self.product_embedding = tf.keras.layers.Embedding(len(vocab_products) + 1, 64, name="candidate_product_emb")
        
        self.marca_lookup = tf.keras.layers.StringLookup(vocabulary=vocab_marca, mask_token=None)
        self.marca_embedding = tf.keras.layers.Embedding(len(vocab_marca) + 1, 16, name="candidate_marca_emb")
        
        self.fam1_lookup = tf.keras.layers.StringLookup(vocabulary=vocab_familia1, mask_token=None)
        self.fam1_embedding = tf.keras.layers.Embedding(len(vocab_familia1) + 1, 16, name="candidate_fam1_emb")
        
        self.fam2_lookup = tf.keras.layers.StringLookup(vocabulary=vocab_familia2, mask_token=None)
        self.fam2_embedding = tf.keras.layers.Embedding(len(vocab_familia2) + 1, 16, name="candidate_fam2_emb")
        
        self.continuous_normalization = tf.keras.layers.Normalization(axis=-1)
        
        self.mlp = tf.keras.Sequential([
            tf.keras.layers.Dense(256, activation="relu"),
            tf.keras.layers.Dropout(dropout_rate),
            tf.keras.layers.Dense(128, activation="relu"),
            tf.keras.layers.Dropout(dropout_rate),
            tf.keras.layers.Dense(embedding_dim, name="candidate_projection")
        ])
        
    def call(self, inputs):
        prod_emb = self.product_embedding(self.product_lookup(inputs["id_producto"]))
        marca_emb = self.marca_embedding(self.marca_lookup(inputs["marca"]))
        fam1_emb = self.fam1_embedding(self.fam1_lookup(inputs["familia1"]))
        fam2_emb = self.fam2_embedding(self.fam2_lookup(inputs["familia2"]))
        
        precio = tf.expand_dims(inputs["precio"], axis=-1)
        peso = tf.expand_dims(inputs["peso_unitario"], axis=-1)
        continuous_features = tf.concat([precio, peso], axis=-1)
        continuous_norm = self.continuous_normalization(continuous_features)
        
        concatenated = tf.concat([prod_emb, marca_emb, fam1_emb, fam2_emb, continuous_norm], axis=-1)
        return tf.math.l2_normalize(self.mlp(concatenated), axis=-1)

query_tower = QueryTower(vocab_ruc, vocab_ciudad, vocab_ruta, vocab_products, embedding_dim=128)
candidate_tower = CandidateTower(vocab_products, vocab_marca, vocab_familia1, vocab_familia2, embedding_dim=128)

# Adaptar normalizaciones
geo_train = train_df.select(["LATITUD", "LONGITUD"]).sample(n=100_000, seed=42).to_numpy().astype(np.float32)
query_tower.geo_normalization.adapt(geo_train)

continuous_train = products_df.select(["precio", "peso_unitario"]).to_numpy().astype(np.float32)
candidate_tower.continuous_normalization.adapt(continuous_train)

# Dummy forward pass
dummy_q = {c: tf.constant([vocab_ruc[0] if c == "RUC" else (vocab_ciudad[0] if c == "CIUDAD" else (vocab_ruta[0] if c == "RUTA" else (vocab_products[0] if c == "COD_PROD" else 0.0)))]) for c in ["RUC", "CIUDAD", "RUTA", "LATITUD", "LONGITUD", "COD_PROD"]}
dummy_q["LATITUD"] = tf.constant([0.0], dtype=tf.float32)
dummy_q["LONGITUD"] = tf.constant([0.0], dtype=tf.float32)

dummy_c = {c: tf.constant([vocab_products[0] if c == "id_producto" else (vocab_marca[0] if c == "marca" else (vocab_familia1[0] if c == "familia1" else (vocab_familia2[0] if c == "familia2" else 0.0)))]) for c in ["id_producto", "marca", "familia1", "familia2", "precio", "peso_unitario"]}
dummy_c["precio"] = tf.constant([0.0], dtype=tf.float32)
dummy_c["peso_unitario"] = tf.constant([0.0], dtype=tf.float32)

_ = query_tower(dummy_q)
_ = candidate_tower(dummy_c)

query_tower.load_weights("models/query_tower.weights.h5")
candidate_tower.load_weights("models/candidate_tower.weights.h5")
print("Pesos del recomendador Two Towers cargados exitosamente.")


Pesos del recomendador Two Towers cargados exitosamente.


## 5. Evaluación de Clientes Fríos (Cold Users)
Evaluamos el rendimiento en clientes de validación que el modelo SVD nunca vio durante su entrenamiento.


In [6]:
# Muestra aleatoria de clientes fríos
sample_cold_users = val_cold_users.sample(n=min(1000, val_cold_users.height), seed=42)
catalog_ids = products_df["id_producto"].to_list()

# 1. Evaluar Surprise SVD en Cold Users
mu = svd.trainset.global_mean
bi = svd.bi
qi = svd.qi
n_factors = svd.n_factors
to_inner_iid = svd.trainset.to_inner_iid

catalog_bi = np.zeros(len(catalog_ids))
catalog_qi = np.zeros((len(catalog_ids), n_factors))
for idx, item_id in enumerate(catalog_ids):
    if svd.trainset.knows_item(item_id):
        inner_id = to_inner_iid(item_id)
        catalog_bi[idx] = bi[inner_id]
        catalog_qi[idx] = qi[inner_id]

def evaluate_svd_cold(df, catalog_ids, k_list=[10, 50, 100]):
    recalls = {k: 0.0 for k in k_list}
    mrr_sum = 0.0
    n = len(df)
    targets = df["COD_PROD_2"].to_numpy()
    default_pu = np.zeros(n_factors) # Factor nulo para usuario frío
    
    for i in range(n):
        target = targets[i]
        # Predicción para usuario frío se limita a: Global Mean + Item Bias
        scores = mu + 0.0 + catalog_bi + np.dot(catalog_qi, default_pu)
        sorted_indices = np.argsort(scores)[::-1]
        sorted_ids = [catalog_ids[idx] for idx in sorted_indices]
        
        try:
            rank = sorted_ids.index(target) + 1
            mrr_sum += 1.0 / rank
            for k in k_list:
                if rank <= k:
                    recalls[k] += 1.0
        except ValueError:
            pass
    return {k: recalls[k] / n for k in k_list}, mrr_sum / n

svd_cold_u_recalls, svd_cold_u_mrr = evaluate_svd_cold(sample_cold_users, catalog_ids)

# 2. Evaluar Two Towers en Cold Users
def evaluate_two_towers_cold(query_tower, candidate_tower, df, catalog_df, k_list=[10, 50, 100]):
    # Precomputar embeddings catálogo
    cand_ds = tf.data.Dataset.from_tensor_slices({
        "id_producto": catalog_df["id_producto"].to_numpy(),
        "marca": catalog_df["marca"].to_numpy(),
        "familia1": catalog_df["familia1"].to_numpy(),
        "familia2": catalog_df["familia2"].to_numpy(),
        "precio": catalog_df["precio"].to_numpy().astype(np.float32),
        "peso_unitario": catalog_df["peso_unitario"].to_numpy().astype(np.float32),
    }).batch(1024)
    cand_embeddings = tf.concat([candidate_tower(b) for b in cand_ds], axis=0)
    
    # Embeddings consultas
    query_ds = tf.data.Dataset.from_tensor_slices({
        "RUC": df["RUC"].to_numpy(),
        "CIUDAD": df["CIUDAD"].to_numpy(),
        "RUTA": df["RUTA"].to_numpy(),
        "LATITUD": df["LATITUD"].to_numpy().astype(np.float32),
        "LONGITUD": df["LONGITUD"].to_numpy().astype(np.float32),
        "COD_PROD": df["COD_PROD"].to_numpy()
    }).batch(1024)
    q_embeddings = tf.concat([query_tower(b) for b in query_ds], axis=0)
    
    similarity = tf.matmul(q_embeddings, cand_embeddings, transpose_b=True).numpy()
    targets = df["COD_PROD_2"].to_numpy()
    catalog_ids_arr = np.array(catalog_df["id_producto"].to_list())
    recalls = {k: 0.0 for k in k_list}
    mrr_sum = 0.0
    n = len(df)
    
    for i in range(n):
        target = targets[i]
        scores = similarity[i]
        top_indices = np.argsort(scores)[::-1]
        sorted_ids = catalog_ids_arr[top_indices]
        
        idx = np.where(sorted_ids == target)[0]
        if len(idx) > 0:
            rank = idx[0] + 1
            mrr_sum += 1.0 / rank
            for k in k_list:
                if rank <= k:
                    recalls[k] += 1.0
                    
    return {k: recalls[k] / n for k in k_list}, mrr_sum / n

tt_cold_u_recalls, tt_cold_u_mrr = evaluate_two_towers_cold(query_tower, candidate_tower, sample_cold_users, products_df)

print("\n--- ARRANCAR EN FRÍO DE CLIENTES (COLD USERS) ---")
print("| Métrica     | Surprise SVD       | Two Towers (LogQ)   |")
print("|:------------|:------------------:|:-------------------:|")
print(f"| Recall@10   | {svd_cold_u_recalls[10]*100:.2f}%              | {tt_cold_u_recalls[10]*100:.2f}%               |")
print(f"| Recall@50   | {svd_cold_u_recalls[50]*100:.2f}%              | {tt_cold_u_recalls[50]*100:.2f}%               |")
print(f"| Recall@100  | {svd_cold_u_recalls[100]*100:.2f}%              | {tt_cold_u_recalls[100]*100:.2f}%               |")
print(f"| MRR         | {svd_cold_u_mrr:.4f}             | {tt_cold_u_mrr:.4f}              |")



--- ARRANCAR EN FRÍO DE CLIENTES (COLD USERS) ---
| Métrica     | Surprise SVD       | Two Towers (LogQ)   |
|:------------|:------------------:|:-------------------:|
| Recall@10   | 0.00%              | 7.70%               |
| Recall@50   | 0.00%              | 17.40%               |
| Recall@100  | 0.00%              | 27.20%               |
| MRR         | 0.0003             | 0.0368              |


## 6. Evaluación de Productos Fríos (Cold Items)
Evaluamos el rendimiento cuando el producto complementario objetivo es un nuevo producto catalogado que no registra transacciones en el set de entrenamiento.


In [7]:
sample_cold_items = val_cold_items.sample(n=min(1000, val_cold_items.height), seed=42)

# 1. Evaluar Surprise SVD en Cold Items
def evaluate_svd_cold_items(df, catalog_ids, k_list=[10, 50, 100]):
    recalls = {k: 0.0 for k in k_list}
    mrr_sum = 0.0
    n = len(df)
    targets = df["COD_PROD_2"].to_numpy()
    to_inner_uid = svd.trainset.to_inner_uid
    
    for i in range(n):
        user_id = df["RUC"][i]
        target = targets[i]
        
        if svd.trainset.knows_user(user_id):
            inner_uid = to_inner_uid(user_id)
            u_bias = svd.bu[inner_uid]
            u_factor = svd.pu[inner_uid]
        else:
            u_bias = 0.0
            u_factor = np.zeros(n_factors)
            
        scores = mu + u_bias + catalog_bi + np.dot(catalog_qi, u_factor)
        sorted_indices = np.argsort(scores)[::-1]
        sorted_ids = [catalog_ids[idx] for idx in sorted_indices]
        
        try:
            # Si el target es un producto frío, no existe en qi/bi ( Surprise le asigna bias por default = 0)
            rank = sorted_ids.index(target) + 1
            mrr_sum += 1.0 / rank
            for k in k_list:
                if rank <= k:
                    recalls[k] += 1.0
        except ValueError:
            pass
    return {k: recalls[k] / n for k in k_list}, mrr_sum / n

svd_cold_i_recalls, svd_cold_i_mrr = evaluate_svd_cold_items(sample_cold_items, catalog_ids)

# 2. Evaluar Two Towers en Cold Items
tt_cold_i_recalls, tt_cold_i_mrr = evaluate_two_towers_cold(query_tower, candidate_tower, sample_cold_items, products_df)

print("\n--- ARRANCAR EN FRÍO DE PRODUCTOS (COLD ITEMS) ---")
print("| Métrica     | Surprise SVD       | Two Towers (LogQ)   |")
print("|:------------|:------------------:|:-------------------:|")
print(f"| Recall@10   | {svd_cold_i_recalls[10]*100:.2f}%              | {tt_cold_i_recalls[10]*100:.2f}%               |")
print(f"| Recall@50   | {svd_cold_i_recalls[50]*100:.2f}%              | {tt_cold_i_recalls[50]*100:.2f}%               |")
print(f"| Recall@100  | {svd_cold_i_recalls[100]*100:.2f}%              | {tt_cold_i_recalls[100]*100:.2f}%               |")
print(f"| MRR         | {svd_cold_i_mrr:.4f}             | {tt_cold_i_mrr:.4f}              |")


# Registro canónico de arranque en frío. Lo consume el cuaderno 16 para su tabla
# comparativa, de modo que ambos cuadernos reporten siempre la misma medición.
import json as _json
_cold = {
    "users": {"Recall@10": tt_cold_u_recalls[10] * 100, "Recall@50": tt_cold_u_recalls[50] * 100,
              "Recall@100": tt_cold_u_recalls[100] * 100, "MRR": tt_cold_u_mrr},
    "items": {"Recall@10": tt_cold_i_recalls[10] * 100, "Recall@50": tt_cold_i_recalls[50] * 100,
              "Recall@100": tt_cold_i_recalls[100] * 100, "MRR": tt_cold_i_mrr},
}
with open("experimentos/results_cold_start.json", "w") as _fh:
    _json.dump(_cold, _fh, indent=2)
print("\nexperimentos/results_cold_start.json actualizado desde este cuaderno.")


--- ARRANCAR EN FRÍO DE PRODUCTOS (COLD ITEMS) ---
| Métrica     | Surprise SVD       | Two Towers (LogQ)   |
|:------------|:------------------:|:-------------------:|
| Recall@10   | 0.00%              | 0.00%               |
| Recall@50   | 0.00%              | 0.40%               |
| Recall@100  | 0.00%              | 0.70%               |
| MRR         | 0.0002             | 0.0006              |

experimentos/results_cold_start.json actualizado desde este cuaderno.


## 7. Estudio de Ablación por Enmascaramiento de Atributos (Inference Masking)
Simulamos la ausencia de variables demográficas y geográficas en inferencia enmascarando los inputs correspondientes de la Query Tower con el token out-of-vocabulary para `RUC` y 0.0 (la media normalizada) para `LATITUD/LONGITUD`.


In [8]:
val_sample_all = val_joined.sample(n=min(2000, val_joined.height), seed=42)

def evaluate_masked_query_tower(query_tower, candidate_tower, df, catalog_df, mask_type="none", k_list=[10, 50, 100]):
    # Precomputar catálogo
    cand_ds = tf.data.Dataset.from_tensor_slices({
        "id_producto": catalog_df["id_producto"].to_numpy(),
        "marca": catalog_df["marca"].to_numpy(),
        "familia1": catalog_df["familia1"].to_numpy(),
        "familia2": catalog_df["familia2"].to_numpy(),
        "precio": catalog_df["precio"].to_numpy().astype(np.float32),
        "peso_unitario": catalog_df["peso_unitario"].to_numpy().astype(np.float32),
    }).batch(1024)
    cand_embeddings = tf.concat([candidate_tower(b) for b in cand_ds], axis=0)
    
    # Generar inputs con máscara
    ruc_input = df["RUC"].to_numpy()
    lat_input = df["LATITUD"].to_numpy().astype(np.float32)
    lon_input = df["LONGITUD"].to_numpy().astype(np.float32)
    prod_input = df["COD_PROD"].to_numpy()
    
    if mask_type == "RUC":
        # Enmascarar RUC con un token que no exista en el vocabulario
        ruc_input = np.array(["CLIENTE_UNKNOWN"] * len(df))
    elif mask_type == "GEO":
        # Enmascarar geolocalización reemplazando con el centro de la normalización (0.0)
        lat_input = np.zeros_like(lat_input)
        lon_input = np.zeros_like(lon_input)
    elif mask_type == "SEED":
        # Enmascarar el producto-semilla con un token fuera del vocabulario: mide
        # cuánto depende la recuperación del producto desde el que parte la consulta.
        prod_input = np.array(["PRODUCTO_UNKNOWN"] * len(df))
        
    query_ds = tf.data.Dataset.from_tensor_slices({
        "RUC": ruc_input,
        "CIUDAD": df["CIUDAD"].to_numpy(),
        "RUTA": df["RUTA"].to_numpy(),
        "LATITUD": lat_input,
        "LONGITUD": lon_input,
        "COD_PROD": prod_input
    }).batch(1024)
    
    q_embeddings = tf.concat([query_tower(b) for b in query_ds], axis=0)
    similarity = tf.matmul(q_embeddings, cand_embeddings, transpose_b=True).numpy()
    targets = df["COD_PROD_2"].to_numpy()
    catalog_ids_arr = np.array(catalog_df["id_producto"].to_list())
    recalls = {k: 0.0 for k in k_list}
    mrr_sum = 0.0
    n = len(df)
    
    for i in range(n):
        target = targets[i]
        scores = similarity[i]
        top_indices = np.argsort(scores)[::-1]
        sorted_ids = catalog_ids_arr[top_indices]
        
        idx = np.where(sorted_ids == target)[0]
        if len(idx) > 0:
            rank = idx[0] + 1
            mrr_sum += 1.0 / rank
            for k in k_list:
                if rank <= k:
                    recalls[k] += 1.0
    return {k: recalls[k] / n for k in k_list}, mrr_sum / n

# Evaluar los tres escenarios
print("Evaluando modelo completo...")
m_full, mrr_full = evaluate_masked_query_tower(query_tower, candidate_tower, val_sample_all, products_df, mask_type="none")
print("Evaluando modelo con RUC enmascarado...")
m_no_ruc, mrr_no_ruc = evaluate_masked_query_tower(query_tower, candidate_tower, val_sample_all, products_df, mask_type="RUC")
print("Evaluando modelo con Coordenadas enmascaradas...")
m_no_geo, mrr_no_geo = evaluate_masked_query_tower(query_tower, candidate_tower, val_sample_all, products_df, mask_type="GEO")

print("Evaluando modelo con producto-semilla enmascarado...")
m_no_seed, mrr_no_seed = evaluate_masked_query_tower(query_tower, candidate_tower, val_sample_all, products_df, mask_type="SEED")

print("\n--- ESTUDIO DE ABLACIÓN (FEATURE MASKING) ---")
print("| Métrica     | Modelo Completo    | Sin Identidad (No RUC) | Sin Geolocalización | Sin Producto-Semilla |")
print("|:------------|:------------------:|:----------------------:|:--------------------:|:--------------------:|")
print(f"| Recall@10   | {m_full[10]*100:.2f}%              | {m_no_ruc[10]*100:.2f}%                | {m_no_geo[10]*100:.2f}%               | {m_no_seed[10]*100:.2f}%               |")
print(f"| Recall@50   | {m_full[50]*100:.2f}%              | {m_no_ruc[50]*100:.2f}%                | {m_no_geo[50]*100:.2f}%               | {m_no_seed[50]*100:.2f}%               |")
print(f"| Recall@100  | {m_full[100]*100:.2f}%              | {m_no_ruc[100]*100:.2f}%                | {m_no_geo[100]*100:.2f}%               | {m_no_seed[100]*100:.2f}%               |")
print(f"| MRR         | {mrr_full:.4f}             | {mrr_no_ruc:.4f}               | {mrr_no_geo:.4f}              | {mrr_no_seed:.4f}              |")


# Registro canónico del estudio de ablación por enmascaramiento de atributos.
_abl = {"completo":      {"Recall@10": m_full[10]*100,    "Recall@50": m_full[50]*100,    "Recall@100": m_full[100]*100,    "MRR": mrr_full},
        "sin_ruc":       {"Recall@10": m_no_ruc[10]*100,  "Recall@50": m_no_ruc[50]*100,  "Recall@100": m_no_ruc[100]*100,  "MRR": mrr_no_ruc},
        "sin_geo":       {"Recall@10": m_no_geo[10]*100,  "Recall@50": m_no_geo[50]*100,  "Recall@100": m_no_geo[100]*100,  "MRR": mrr_no_geo},
        "sin_semilla":   {"Recall@10": m_no_seed[10]*100, "Recall@50": m_no_seed[50]*100, "Recall@100": m_no_seed[100]*100, "MRR": mrr_no_seed}}
with open("experimentos/results_feature_ablation.json", "w") as _fh:
    _json.dump(_abl, _fh, indent=2)
print("\nexperimentos/results_feature_ablation.json actualizado desde este cuaderno.")

Evaluando modelo completo...


Evaluando modelo con RUC enmascarado...


Evaluando modelo con Coordenadas enmascaradas...


Evaluando modelo con producto-semilla enmascarado...



--- ESTUDIO DE ABLACIÓN (FEATURE MASKING) ---
| Métrica     | Modelo Completo    | Sin Identidad (No RUC) | Sin Geolocalización | Sin Producto-Semilla |
|:------------|:------------------:|:----------------------:|:--------------------:|:--------------------:|
| Recall@10   | 7.40%              | 5.25%                | 1.20%               | 7.15%               |
| Recall@50   | 21.50%              | 15.45%                | 6.75%               | 20.80%               |
| Recall@100  | 33.05%              | 23.85%                | 14.10%               | 32.40%               |
| MRR         | 0.0354             | 0.0265               | 0.0079              | 0.0321              |

experimentos/results_feature_ablation.json actualizado desde este cuaderno.


## Conclusiones del Experimento

1. **Generalización de la Estructura de Torres**: Mientras que el modelo SVD colaborativo degrada a cero predictivo bajo arranque en frío de productos (debido a su naturaleza basada en embeddings puramente latentes sin atributos), la red Two Towers puede mapear productos nuevos al espacio latente utilizando su marca, precio, peso y familias comerciales.
2. **Mitigación de Usuarios Fríos**: Two Towers retiene capacidad predictiva relevante para nuevos clientes gracias al uso combinado del contexto inmediato (el ítem disparador `COD_PROD` y la ubicación geográfica).
3. **Análisis de Importancia (Ablación)**: Omitir la geolocalización o la identidad individual del cliente demuestra que la mayor parte de la señal en la recomendación complementaria ferretera B2B está dada por la complementariedad técnica (el producto semilla disparador), pero la geolocalización aporta un refino geográfico de mercado crucial.
